# RAG Pipeline Experiments

End‑to‑end testing of the Multimodal RAG assistant.

In [ ]:
import sys
sys.path.append("..")

import os
import tempfile
from rag.rag_chain import RAGChain
from PIL import Image
import matplotlib.pyplot as plt
from utils.data_loader import load_product_data

# Load data once to ensure everything is ready
df = load_product_data()
rag = RAGChain()

### Text‑only Query

In [ ]:
# Simple text query
query = "Show me casual shirts under $50"
result = rag.run(text_query=query, k=3)

print("=== Response ===")
print(result['response'])
print("\n=== Retrieved Products ===")
for item in result['retrieved_products']:
    meta = item['metadata']
    print(f"- {meta['title']} (${meta['price']})")

### Image‑only Query (requires an image file)

In [ ]:
# If you have a test image in the raw/images folder, use the first one available
import glob
image_dir = "data/raw/images"
image_files = glob.glob(f"{image_dir}/*.jpg") + glob.glob(f"{image_dir}/*.png")
if image_files:
    test_image = image_files[0]
    print(f"Using image: {test_image}")
    result = rag.run(image_path=test_image, k=3)
    print("=== Response ===")
    print(result['response'])
    print("\n=== Image Caption ===")
    print(result['image_caption'])
else:
    print("No image files found for testing.")

### Hybrid Query (Text + Image)

In [ ]:
# If we have an image, combine with text
if image_files:
    test_image = image_files[0]
    query = "I need something similar but in blue"
    result = rag.run(text_query=query, image_path=test_image, k=3)
    print("=== Hybrid Query Response ===")
    print(result['response'])

### Performance: Measure Latency

In [ ]:
import time
queries = ["shirt", "jeans", "watch", "sneakers", "bag"]
times = []
for q in queries:
    start = time.time()
    rag.run(text_query=q, k=2)
    times.append(time.time() - start)
print(f"Average latency: {np.mean(times):.2f} seconds")
plt.bar(queries, times)
plt.ylabel("Time (s)")
plt.title("RAG Latency per Query")
plt.show()

### Inspect the Context Built from Retrieved Products

In [ ]:
# Run a query and examine the context
result = rag.run(text_query="black formal shoes", k=2)
print("Context passed to LLM:\n")
print(result['context'])